### Исходные данные

In [24]:
import numpy as np
from numba import njit
import pandas as pd
import time

package_sizes = np.array([25.0, 200.0, 100.0])
commissions = np.array([0.04, 0.07, 0.05])
min_investments = np.array([30.0, 150.0, 100.0])

# Вероятности по этапам 
probabilities = np.array([
    [0.60, 0.30, 0.10],
    [0.30, 0.20, 0.50],
    [0.40, 0.40, 0.20],
])

# Коэффициенты
price_changes = np.array([
    [[1.20, 1.10, 1.07], [1.05, 1.02, 1.03], [0.80, 0.95, 1.00]],
    [[1.40, 1.15, 1.01], [1.05, 1.00, 1.00], [0.60, 0.90, 1.00]],
    [[1.15, 1.12, 1.05], [1.05, 1.01, 1.01], [0.70, 0.94, 1.00]],
])

initial_portfolio = np.array([100.0, 800.0, 400.0, 600.0])
eps = 1e-7  

average_last = probabilities[2] @ price_changes[2]

print('Начальный капитал:', initial_portfolio.sum())
print('Средние коэффициенты третьего этапа:', average_last)


Начальный капитал: 1900.0
Средние коэффициенты третьего этапа: [1.02  1.04  1.024]


### Покупка, продажа и проверка ограничений



In [25]:
@njit
def total_capital(portfolio):
    return portfolio[0] + portfolio[1] + portfolio[2] + portfolio[3]


@njit
def make_trades(portfolio, trades):
    updated = portfolio.copy()
    for asset in range(3):
        amount = package_sizes[asset] * trades[asset]
        fee = commissions[asset] * abs(amount)
        updated[asset] += amount
        updated[3] -= amount + fee
    return updated


@njit
def trades_allowed(portfolio, trades):
    updated = make_trades(portfolio, trades)
    for asset in range(3):
        if updated[asset] < min_investments[asset] - eps:
            return False
    return updated[3] >= -eps


@njit
def market_change(portfolio_after_trades, stage, situation):
    updated = portfolio_after_trades.copy()
    for asset in range(3):
        updated[asset] *= price_changes[stage, situation, asset]
    return updated

@njit
def minimum_packages(portfolio, asset):
    difference = min_investments[asset] - portfolio[asset]
    return int(np.ceil((difference - eps) / package_sizes[asset]))


@njit
def expected_last_capital(portfolio_after_trades):
    result = portfolio_after_trades[3]
    for asset in range(3):
        result += portfolio_after_trades[asset] * average_last[asset]
    return result


example_trades = np.array([2, -1, 0])
example_result = make_trades(initial_portfolio, example_trades)

print('После сделок:', example_result)
print('Сделка допустима:', trades_allowed(initial_portfolio, example_trades))

После сделок: [150. 600. 400. 734.]
Сделка допустима: True


### Границы перебора

In [26]:
@njit
def trade_limits(portfolio):
    lower = np.zeros(3, dtype=np.int64)
    upper = np.zeros(3, dtype=np.int64)
    money_total = total_capital(portfolio)

    for asset in range(3):
        # нижняя граница
        lower[asset] = minimum_packages(portfolio, asset)

        # верхняя граница
        other_min = 0.0
        for other in range(3):
            if other != asset:
                other_min += min_investments[other]

        available = money_total - other_min - min_investments[asset] + portfolio[asset]
        upper[asset] = int(np.floor((available + eps) / package_sizes[asset]))
    return lower, upper


limits_min, limits_max = trade_limits(initial_portfolio)
print('Нижние границы:', limits_min)
print('Верхние границы:', limits_max)


Нижние границы: [-2 -3 -3]
Верхние границы: [68 12 20]


### Третий этап — начало обратного расчёта

Упрощение: если на третьем этапе все минимумы соблюдены, не совершаем добровольных сделок.
При недостатке актива докупаем пакеты до минимума; если денег недостаточно, рассматриваем продажи других активов.
Оно применимо здесь, поскольку ожидаемый выигрыш от покупки меньше комиссии, а продажа уменьшает ожидаемый капитал.
Корректность упрощения дополнительно проверяется независимым перебором на выбранных состояниях ниже.


In [27]:
@njit
def bellman_stage3(portfolio):
    # Минимальные обязательные покупки
    required = np.zeros(3, dtype=np.int64)
    money_needed = 0.0
    for asset in range(3):
        required[asset] = max(0, minimum_packages(portfolio, asset))
        money_needed += package_sizes[asset] * (1 + commissions[asset]) * required[asset]

    if portfolio[3] + eps >= money_needed:
        final_portfolio = make_trades(portfolio, required)
        return expected_last_capital(final_portfolio), required

    deficit = money_needed - portfolio[3]
    sell_limits = np.zeros(3, dtype=np.int64)
    for asset in range(3):
        if required[asset] == 0:
            possible = max(0, int(np.floor(
                (portfolio[asset] - min_investments[asset] + eps) / package_sizes[asset]
            )))
            enough = int(np.ceil(deficit / (package_sizes[asset] * (1 - commissions[asset])))) + 1
            sell_limits[asset] = min(possible, enough)

    best_value = -np.inf
    best_trades = np.zeros(3, dtype=np.int64)
    for sold1 in range(sell_limits[0] + 1):
        for sold2 in range(sell_limits[1] + 1):
            for sold3 in range(sell_limits[2] + 1):
                trades = np.array((required[0] - sold1,
                                   required[1] - sold2,
                                   required[2] - sold3), dtype=np.int64)
                if not trades_allowed(portfolio, trades):
                    continue
                value = expected_last_capital(make_trades(portfolio, trades))
                if value > best_value + 1e-9:
                    best_value = value
                    best_trades = trades
    return best_value, best_trades


### рекурсия Беллмана

Для каждого допустимого управления смотрим три возможных исхода и берём средний результат с учётом вероятностей. 

In [28]:
@njit
def bellman_stage2(portfolio):
    lower, upper = trade_limits(portfolio)
    best_value = -np.inf
    best_trades = np.zeros(3, dtype=np.int64)

    # Перебор количества пакетов каждого актива
    for packages1 in range(lower[0], upper[0] + 1):
        for packages2 in range(lower[1], upper[1] + 1):
            for packages3 in range(lower[2], upper[2] + 1):
                trades = np.array((packages1, packages2, packages3), dtype=np.int64)
                if not trades_allowed(portfolio, trades):
                    continue

                after_trades = make_trades(portfolio, trades)
                average_result = 0.0

                # Три рыночные ситуации второго этапа
                for situation in range(3):
                    next_portfolio = market_change(after_trades, 1, situation)
                    stage3_value, _ = bellman_stage3(next_portfolio)
                    if not np.isfinite(stage3_value):
                        average_result = -np.inf
                        break
                    average_result += probabilities[1, situation] * stage3_value

                if average_result > best_value + 1e-9:
                    best_value = average_result
                    best_trades = trades

    return best_value, best_trades


@njit
def bellman_stage1(portfolio):
    lower, upper = trade_limits(portfolio)
    best_value = -np.inf
    best_trades = np.zeros(3, dtype=np.int64)

    for packages1 in range(lower[0], upper[0] + 1):
        for packages2 in range(lower[1], upper[1] + 1):
            for packages3 in range(lower[2], upper[2] + 1):
                trades = np.array((packages1, packages2, packages3), dtype=np.int64)
                if not trades_allowed(portfolio, trades):
                    continue

                after_trades = make_trades(portfolio, trades)
                average_result = 0.0

                # Три рыночные ситуации первого этапа
                for situation in range(3):
                    next_portfolio = market_change(after_trades, 0, situation)
                    stage2_value, _ = bellman_stage2(next_portfolio)
                    if not np.isfinite(stage2_value):
                        average_result = -np.inf
                        break
                    average_result += probabilities[0, situation] * stage2_value

                if average_result > best_value + 1e-9:
                    best_value = average_result
                    best_trades = trades

    return best_value, best_trades


### Оптимальное первое управление


In [29]:
start_time = time.perf_counter()
optimal_capital, first_trades = bellman_stage1(initial_portfolio)
calculation_time = time.perf_counter() - start_time

print('Первое управление (ЦБ1, ЦБ2, депозит):', tuple(first_trades))
print(f'Ожидаемый конечный капитал: {optimal_capital:.6f} д.е.')
print(f'Ожидаемый доход: {optimal_capital - total_capital(initial_portfolio):.6f} д.е.')
print(f'Время расчёта с компиляцией: {calculation_time:.2f} с')


Первое управление (ЦБ1, ЦБ2, депозит): (np.int64(23), np.int64(0), np.int64(0))
Ожидаемый конечный капитал: 2033.988884 д.е.
Ожидаемый доход: 133.988884 д.е.
Время расчёта с компиляцией: 10.41 с


### Прямой проход — восстановление стратегии

Для каждого из трёх исходов считаем действия второго этапа, после этого — третьего. Получаем **9 ветвей управлений** и **27 конечных исходов**.


In [30]:
situation_names = ['Благоприятная', 'Нейтральная', 'Негативная']
strategy_rows = []
end_rows = []

portfolio_after_first = make_trades(initial_portfolio, first_trades)
assert trades_allowed(initial_portfolio, first_trades)

for first_situation in range(3):
    portfolio2 = market_change(portfolio_after_first, 0, first_situation)
    value2, trades2 = bellman_stage2(portfolio2)
    assert trades_allowed(portfolio2, trades2)
    portfolio_after_second = make_trades(portfolio2, trades2)

    for second_situation in range(3):
        portfolio3 = market_change(portfolio_after_second, 1, second_situation)
        value3, trades3 = bellman_stage3(portfolio3)
        assert np.isfinite(value3) and trades_allowed(portfolio3, trades3)
        portfolio_after_third = make_trades(portfolio3, trades3)

        strategy_rows.append({
            'Исход 1': situation_names[first_situation],
            'Исход 2': situation_names[second_situation],
            'Управление 2': tuple(int(x) for x in trades2),
            'Управление 3': tuple(int(x) for x in trades3),
            'W3': round(value3, 6),
        })

        for third_situation in range(3):
            final_portfolio = market_change(portfolio_after_third, 2, third_situation)
            chance = (probabilities[0, first_situation] *
                      probabilities[1, second_situation] *
                      probabilities[2, third_situation])
            end_rows.append({
                'Исход 1': situation_names[first_situation],
                'Исход 2': situation_names[second_situation],
                'Исход 3': situation_names[third_situation],
                'Вероятность': chance,
                'Капитал': total_capital(final_portfolio),
            })

strategy_table = pd.DataFrame(strategy_rows)
end_table = pd.DataFrame(end_rows)
print('Первое управление:', tuple(first_trades))
display(strategy_table)


Первое управление: (np.int64(23), np.int64(0), np.int64(0))


,Исход 1,Исход 2,Управление 2,Управление 3,W3
0,Благоприятная,Благоприятная,"(-31, 0, 0)","(0, 0, 0)",2291.11472
1,Благоприятная,Нейтральная,"(-31, 0, 0)","(0, 0, 0)",2136.95700
2,Благоприятная,Негативная,"(-31, 0, 0)","(1, 0, 0)",2028.87200
3,Нейтральная,Благоприятная,"(-27, 0, 0)","(0, 0, 0)",2100.23788
4,Нейтральная,Нейтральная,"(-27, 0, 0)","(0, 0, 0)",1956.67425
5,Нейтральная,Негативная,"(-27, 0, 0)","(1, 0, 0)",1855.81900
6,Негативная,Благоприятная,"(-20, 0, 0)","(0, 0, 0)",1861.77600
7,Негативная,Нейтральная,"(-20, 0, 0)","(0, 0, 0)",1724.84000
8,Негативная,Негативная,"(-20, 0, 0)","(1, 0, 0)",1626.94000


### Проверка

Умножаем капитал в каждой конечной ветви на вероятность этой ветви. Сумма должна совпадать со значением Беллмана, найденным в обратном расчёте.

In [31]:
expected_from_branches = (end_table['Вероятность'] * end_table['Капитал']).sum()
print('Сумма вероятностей:', end_table['Вероятность'].sum())
print('По Беллману:', round(optimal_capital, 6))
print('По всем ветвям:', round(expected_from_branches, 6))
assert np.isclose(expected_from_branches, optimal_capital, atol=1e-6)
display(end_table.round(6))


Сумма вероятностей: 1.0
По Беллману: 2033.988884
По всем ветвям: 2033.988884


,Исход 1,Исход 2,Исход 3,Вероятность,Капитал
0,Благоприятная,Благоприятная,Благоприятная,0.072,2389.684000
1,Благоприятная,Благоприятная,Нейтральная,0.072,2256.172800
2,Благоприятная,Благоприятная,Негативная,0.036,2163.860000
3,Благоприятная,Нейтральная,Благоприятная,0.048,2223.262500
4,Благоприятная,Нейтральная,Нейтральная,0.048,2105.667500
5,Благоприятная,Нейтральная,Негативная,0.024,2026.925000
6,Благоприятная,Негативная,Благоприятная,0.120,2109.340000
7,Благоприятная,Негативная,Нейтральная,0.120,2000.500000
8,Благоприятная,Негативная,Негативная,0.060,1924.680000
9,Нейтральная,Благоприятная,Благоприятная,0.036,2192.271500


### Таблицы для отчёта: сравнение управлений второго и первого этапов


In [32]:
def evaluate_stage2_choice(state, action):
    action = np.array(action, dtype=np.int64)
    if not trades_allowed(state, action):
        return -np.inf
    after = make_trades(state, action)
    answer = 0.0
    for k in range(3):
        next_state = market_change(after, 1, k)
        value, _ = bellman_stage3(next_state)
        if not np.isfinite(value):
            return -np.inf
        answer += probabilities[1, k] * value
    return answer

state2 = market_change(make_trades(initial_portfolio, first_trades), 0, 0)
options2 = [(-31,0,0), (-30,0,0), (-29,0,0), (-31,0,1), (-31,0,-1)]
table8 = pd.DataFrame([
    {'Управление 2': a, 'Ожидаемый капитал': evaluate_stage2_choice(state2, a)}
    for a in options2
])
print('Таблица 8:')
display(table8.round(6))

def evaluate_stage1_choice(state, action):
    action = np.array(action, dtype=np.int64)
    if not trades_allowed(state, action):
        return -np.inf
    after = make_trades(state, action)
    answer = 0.0
    for k in range(3):
        next_state = market_change(after, 0, k)
        value, _ = bellman_stage2(next_state)
        if not np.isfinite(value):
            return -np.inf
        answer += probabilities[0, k] * value
    return answer

lower, upper = trade_limits(initial_portfolio)
all_first = []
for a in range(int(lower[0]), int(upper[0]) + 1):
    for b in range(int(lower[1]), int(upper[1]) + 1):
        for c in range(int(lower[2]), int(upper[2]) + 1):
            trade = np.array([a, b, c], dtype=np.int64)
            if trades_allowed(initial_portfolio, trade):
                all_first.append((a, b, c))

print('Допустимых управлений первого этапа:', len(all_first))

@njit
def score_all_first(states, actions):
    values = np.empty(len(actions), dtype=np.float64)
    for i in range(len(actions)):
        after = make_trades(states, actions[i])
        result = 0.0
        for situation in range(3):
            next_state = market_change(after, 0, situation)
            value, _ = bellman_stage2(next_state)
            if not np.isfinite(value):
                result = -np.inf
                break
            result += probabilities[0, situation] * value
        values[i] = result
    return values

first_array = np.array(all_first, dtype=np.int64)
first_scores = score_all_first(initial_portfolio, first_array)
ranked = [(a, float(v)) for a, v in zip(all_first, first_scores) if np.isfinite(v)]
ranked.sort(key=lambda row: (-row[1], row[0]))
table10_top = pd.DataFrame(ranked[:5], columns=['Управление 1', 'Ожидаемый капитал'])
print('Таблица 10: пять лучших допустимых управлений')
display(table10_top.round(6))

no_trade_value = dict(ranked).get((0, 0, 0), -np.inf)
print('Без сделок:', no_trade_value)
print('Совпадение максимума с Беллманом:', np.isclose(ranked[0][1], optimal_capital, atol=1e-6))
assert np.isclose(ranked[0][1], optimal_capital, atol=1e-6)
assert ranked[0][0] == tuple(int(x) for x in first_trades)

tie_tolerance = 1e-9
best_value = ranked[0][1]
best_actions = [action for action, value in ranked if abs(value - best_value) <= tie_tolerance]
print('Количество оптимальных первых управлений:', len(best_actions))
print('Оптимальные первые управления:', best_actions)
if len(ranked) > 1:
    print('Отрыв от второго места:', round(best_value - ranked[1][1], 9))
print('При равных результатах выбирается первое найденное управление.')
print('Перебор выполняется по возрастанию количества пакетов.')


Таблица 8:


,Управление 2,Ожидаемый капитал
0,"(-31, 0, 0)",2129.161816
1,"(-30, 0, 0)",2129.126816
2,"(-29, 0, 0)",2128.841816
3,"(-31, 0, 1)",2126.869016
4,"(-31, 0, -1)",2121.454616


Допустимых управлений первого этапа: 1502
Таблица 10: пять лучших допустимых управлений


,Управление 1,Ожидаемый капитал
0,"(23, 0, 0)",2033.988884
1,"(19, 0, 1)",2033.872251
2,"(15, 0, 2)",2033.691118
3,"(22, 0, 0)",2033.261659
4,"(18, 0, 1)",2033.145026


Без сделок: 2016.4647088000002
Совпадение максимума с Беллманом: True
Количество оптимальных первых управлений: 1
Оптимальные первые управления: [(23, 0, 0)]
Отрыв от второго места: 0.1166328
При равных результатах выбирается первое найденное управление.
Перебор выполняется по возрастанию количества пакетов.


### Независимая проверка третьего этапа полным перебором




In [33]:
def exhaustive_last_stage(state):
    sizes = [25.0, 200.0, 100.0]
    fees = [0.04, 0.07, 0.05]
    minimums = [30.0, 150.0, 100.0]
    avg = [1.020, 1.040, 1.024]
    lower = [int(np.ceil((minimums[j] - state[j] - 1e-9) / sizes[j])) for j in range(3)]
    upper = [int(np.floor((sum(state) / sizes[j]))) for j in range(3)]
    best = -float('inf')
    optimal = []
    for n1 in range(lower[0], upper[0] + 1):
        for n2 in range(lower[1], upper[1] + 1):
            for n3 in range(lower[2], upper[2] + 1):
                n = (n1, n2, n3)
                amounts = [state[j] + sizes[j] * n[j] for j in range(3)]
                cash = state[3] - sum(sizes[j] * n[j] + fees[j] * sizes[j] * abs(n[j]) for j in range(3))
                if cash < -1e-8 or any(amounts[j] < minimums[j] - 1e-8 for j in range(3)):
                    continue
                result = cash + sum(amounts[j] * avg[j] for j in range(3))
                if result > best + 1e-7:
                    best = result
                    optimal = [n]
                elif abs(result - best) <= 1e-7:
                    optimal.append(n)
    return best, optimal

states_to_check = [
    ('Все минимумы выполнены', np.array([49., 1012., 432.28, 746.])),
    ('Нужна покупка ЦБ1', np.array([21., 792., 428., 746.])),
    ('Недостаточно денег для обязательной покупки', np.array([21., 792., 428., 0.])),
]
last_stage_test_rows = []
for label, state in states_to_check:
    expected_value, expected_actions = exhaustive_last_stage(state)
    actual_value, actual_action = bellman_stage3(state)
    ok = (np.isclose(expected_value, actual_value, atol=1e-6) and
          tuple(int(v) for v in actual_action) in expected_actions)
    last_stage_test_rows.append({
        'Тест': label, 'Вход S3': tuple(state),
        'Ожидание (перебор)': expected_value,
        'Факт (Беллман)': actual_value,
        'Статус': 'OK' if ok else 'ОШИБКА'
    })
assert all(r['Статус'] == 'OK' for r in last_stage_test_rows)
display(pd.DataFrame(last_stage_test_rows).round(6))


,Тест,Вход S3,Ожидание (перебор),Факт (Беллман),Статус
0,Все минимумы выполнены,"(49.0, 1012.0, 432.28, 746.0)",2291.11472,2291.11472,OK
1,Нужна покупка ЦБ1,"(21.0, 792.0, 428.0, 746.0)",2028.87200,2028.87200,OK
2,Недостаточно денег для обязательной покупки,"(21.0, 792.0, 428.0, 0.0)",1275.47200,1275.47200,OK


### Независимый полный перебор на маленькой задаче

Берём отдельную упрощённую модель: **один актив, два этапа, по два исхода на каждом этапе**, пакеты по 10 д.е., комиссии 0, минимальное вложение 0, начальное состояние (10 д.е. в активе, 10 д.е. свободными). Сначала находим максимум методом Беллмана (выбор лучшего продолжения для каждого состояния), а затем **независимо** перебираем все стратегии (первую сделку и обе возможные сделки второго этапа). Это тест именно логики динамического программирования, а не подстановка прежнего ответа.


In [34]:
from itertools import product

toy_prices = ((1.5, 0.5), (1.4, 1.0))
toy_prob = ((0.5, 0.5), (0.5, 0.5))

def toy_actions(asset, cash):
    return [n for n in range(-int(asset // 10), int(cash // 10) + 1)]

def toy_next(asset, cash, n, factor):
    return (asset + 10 * n) * factor, cash - 10 * n

def toy_dp2(asset, cash):
    values = []
    for n in toy_actions(asset, cash):
        result = sum(toy_prob[1][k] * sum(toy_next(asset, cash, n, toy_prices[1][k]))
                     for k in range(2))
        values.append(result)
    return max(values)

def toy_dp1(asset, cash):
    return max(sum(toy_prob[0][k] * toy_dp2(*toy_next(asset, cash, n, toy_prices[0][k]))
                   for k in range(2)) for n in toy_actions(asset, cash))

def toy_all_strategies(asset, cash):
    results = []
    for first in toy_actions(asset, cash):
        next0 = toy_next(asset, cash, first, toy_prices[0][0])
        next1 = toy_next(asset, cash, first, toy_prices[0][1])
        for second0, second1 in product(toy_actions(*next0), toy_actions(*next1)):
            final_expected = 0.0
            for k, second in enumerate((second0, second1)):
                stage2_asset, stage2_cash = (next0, next1)[k]
                for m in range(2):
                    final_asset, final_cash = toy_next(stage2_asset, stage2_cash, second, toy_prices[1][m])
                    final_expected += toy_prob[0][k] * toy_prob[1][m] * (final_asset + final_cash)
            results.append(((first, second0, second1), final_expected))
    return max(results, key=lambda x: x[1])

toy_dp = toy_dp1(10., 10.)
toy_strategy, toy_brute = toy_all_strategies(10., 10.)
print('Вход: один актив, 2 этапа, портфель (10, 10), пакет 10, комиссия 0')
print('Ожидание (независимый перебор):', toy_brute)
print('Факт (динамическое программирование):', toy_dp)
print('Одна из оптимальных стратегий перебора:', toy_strategy)
assert np.isclose(toy_dp, toy_brute, atol=1e-9)
print('Результат: OK')


Вход: один актив, 2 этапа, портфель (10, 10), пакет 10, комиссия 0
Ожидание (независимый перебор): 24.0
Факт (динамическое программирование): 24.0
Одна из оптимальных стратегий перебора: (-1, 2, 2)
Результат: OK


### Граничные случаи и некорректные входные данные



In [35]:
def check_input(portfolio, sizes, fees, minimums, probs, changes):
    portfolio = np.asarray(portfolio, dtype=float)
    sizes = np.asarray(sizes, dtype=float)
    fees = np.asarray(fees, dtype=float)
    minimums = np.asarray(minimums, dtype=float)
    probs = np.asarray(probs, dtype=float)
    changes = np.asarray(changes, dtype=float)

    if (portfolio.shape != (4,) or sizes.shape != (3,)
            or fees.shape != (3,) or minimums.shape != (3,)
            or probs.ndim != 2
            or changes.shape != (*probs.shape, 3)):
        raise ValueError("Неверные размеры данных")

    for data in (portfolio, sizes, fees, minimums, probs, changes):
        if not np.all(np.isfinite(data)):
            raise ValueError("Некорректные числа")

    if np.any(portfolio < 0) or np.any(minimums < 0):
        raise ValueError("Отрицательные суммы")

    if np.any(sizes <= 0):
        raise ValueError("Размеры пакетов должны быть положительными")

    if np.any(fees < 0) or np.any(fees >= 1):
        raise ValueError("Некорректные комиссии")

    if np.any(probs < 0) or not np.allclose(probs.sum(axis=1), 1):
        raise ValueError("Некорректные вероятности")

    if np.any(changes < 0):
        raise ValueError("Отрицательные коэффициенты")

    return True


# тестирование
tests = []


def add_test(name, data, expected, actual, correct):
    tests.append([
        name,
        str(data),
        str(expected),
        str(actual),
        bool(correct)
    ])


zero_result = toy_dp1(0, 0)

add_test(
    "Нулевой портфель",
    "Актив=0, деньги=0, минимумы=0",
    0,
    zero_result,
    np.isclose(zero_result, 0)
)


values = [
    sum(toy_next(10, 10, n, 1.0))
    for n in toy_actions(10, 10)
]

add_test(
    "Нулевая доходность",
    "Актив=10, деньги=10, r=1, комиссия=0",
    "Капитал 20 при любой сделке",
    values,
    all(np.isclose(v, 20) for v in values)
)


add_test(
    "Несколько оптимумов",
    "Актив=10, деньги=10, r=1, комиссия=0",
    "Не менее двух оптимумов",
    len(values),
    len(values) >= 2 and np.allclose(values, values[0])
)


sale = np.array([-3, 0, 0], dtype=np.int64)
sale_allowed = trades_allowed(initial_portfolio, sale)

add_test(
    "Минимальные вложения",
    "S1=(100,800,400,600), U=(-3,0,0)",
    False,
    sale_allowed,
    not sale_allowed
)


buy = np.array([24, 0, 0], dtype=np.int64)
buy_allowed = trades_allowed(initial_portfolio, buy)

add_test(
    "Запрет кредита",
    "S1=(100,800,400,600), U=(24,0,0)",
    False,
    buy_allowed,
    not buy_allowed
)


valid = check_input(
    initial_portfolio,
    package_sizes,
    commissions,
    min_investments,
    probabilities,
    price_changes
)

add_test(
    "Корректные данные",
    "Исходные параметры задачи",
    True,
    valid,
    valid
)


bad_cases = [
    (
        "Отрицательная сумма",
        "Первый актив = -1",
        [-1, 800, 400, 600],
        probabilities,
        commissions
    ),
    (
        "Неверные вероятности",
        "Вероятности = (0.6, 0.3, 0.2)",
        initial_portfolio,
        np.array([
            [0.6, 0.3, 0.2],
            probabilities[1],
            probabilities[2]
        ]),
        commissions
    ),
    (
        "Отрицательная комиссия",
        "Комиссия ЦБ1 = -0.04",
        initial_portfolio,
        probabilities,
        [-0.04, 0.07, 0.05]
    )
]

for name, data, portfolio, probs, fees in bad_cases:
    try:
        check_input(
            portfolio,
            package_sizes,
            fees,
            min_investments,
            probs,
            price_changes
        )
        actual = "Ошибка не обнаружена"
    except ValueError:
        actual = "ValueError"

    add_test(
        name,
        data,
        "ValueError",
        actual,
        actual == "ValueError"
    )


manual_tests = [
    ([21, 792, 428, 746], 2028.872, (1, 0, 0), 3),
    ([810, 880, 428, 2], 2129.161816, (-31, 0, 0), 2),
    ([708.75, 816, 412, 2], 1949.315714, (-27, 0, 0), 2),
    ([540, 760, 400, 2], 1716.9708, (-20, 0, 0), 2)
]

for state, expected, action, stage in manual_tests:
    state_array = np.array(state, dtype=float)

    if stage == 3:
        value, found = bellman_stage3(state_array)
    else:
        value, found = bellman_stage2(state_array)

    correct = (
        np.isclose(value, expected, atol=1e-6)
        and tuple(found) == action
    )

    add_test(
        f"Ручной расчёт W{stage}",
        f"S={state}",
        f"{expected:.6f}, U={action}",
        f"{value:.6f}, U={tuple(found)}",
        correct
    )


results = pd.DataFrame(
    tests,
    columns=[
        "Тест",
        "Входные данные",
        "Ожидаемый результат",
        "Фактический результат",
        "Успешно"
    ]
)

display(results)

print("Пройдено тестов:", results["Успешно"].sum(), "из", len(results))

assert results["Успешно"].all()

,Тест,Входные данные,Ожидаемый результат,Фактический результат,Успешно
0,Нулевой портфель,"Актив=0, деньги=0, минимумы=0",0,0.0,True
1,Нулевая доходность,"Актив=10, деньги=10, r=1, комиссия=0",Капитал 20 при любой сделке,"[20.0, 20.0, 20.0]",True
2,Несколько оптимумов,"Актив=10, деньги=10, r=1, комиссия=0",Не менее двух оптимумов,3,True
3,Минимальные вложения,"S1=(100,800,400,600), U=(-3,0,0)",False,False,True
4,Запрет кредита,"S1=(100,800,400,600), U=(24,0,0)",False,False,True
5,Корректные данные,Исходные параметры задачи,True,True,True
6,Отрицательная сумма,Первый актив = -1,ValueError,ValueError,True
7,Неверные вероятности,"Вероятности = (0.6, 0.3, 0.2)",ValueError,ValueError,True
8,Отрицательная комиссия,Комиссия ЦБ1 = -0.04,ValueError,ValueError,True
9,Ручной расчёт W3,"S=[21, 792, 428, 746]","2028.872000, U=(1, 0, 0)","2028.872000, U=(np.int64(1), np.int64(0), np.i...",True


Пройдено тестов: 13 из 13


### Дополнительная проверка основного алгоритма

Проверим второй этап двумя способами: основной функцией
Беллмана и независимым полным перебором.


In [36]:
def check_second_stage(state):
    sizes = [25, 200, 100]
    fees = [0.04, 0.07, 0.05]
    minimums = [30, 150, 100]

    probs = [0.3, 0.2, 0.5]

    changes = [
        [1.40, 1.15, 1.01],
        [1.05, 1.00, 1.00],
        [0.60, 0.90, 1.00]
    ]

    lower = [
        int(np.ceil((minimums[j] - state[j] - 1e-9) / sizes[j]))
        for j in range(3)
    ]

    upper = []

    for j in range(3):
        other_minimums = sum(
            minimums[k] for k in range(3) if k != j
        )

        available = sum(state) - other_minimums - state[j]

        upper.append(
            int(np.floor((available + 1e-9) / sizes[j]))
        )

    best_value = -np.inf
    best_actions = []
    checked = 0

    for n1 in range(lower[0], upper[0] + 1):
        for n2 in range(lower[1], upper[1] + 1):
            for n3 in range(lower[2], upper[2] + 1):

                action = (n1, n2, n3)

                assets = [
                    state[j] + sizes[j] * action[j]
                    for j in range(3)
                ]

                cash = state[3] - sum(
                    sizes[j] * action[j]
                    + fees[j] * sizes[j] * abs(action[j])
                    for j in range(3)
                )

                if cash < -1e-8:
                    continue

                if any(
                    assets[j] < minimums[j] - 1e-8
                    for j in range(3)
                ):
                    continue

                checked += 1
                expected = 0

                for k in range(3):

                    next_state = [
                        assets[j] * changes[k][j]
                        for j in range(3)
                    ] + [cash]

                    # Независимый перебор третьего этапа
                    value, _ = exhaustive_last_stage(next_state)

                    if not np.isfinite(value):
                        expected = -np.inf
                        break

                    expected += probs[k] * value

                if expected > best_value + 1e-7:
                    best_value = expected
                    best_actions = [action]

                elif (
                    np.isfinite(expected)
                    and abs(expected - best_value) <= 1e-7
                ):
                    best_actions.append(action)

    return best_value, best_actions, checked


small_states = [
    np.array([50., 200., 150., 50.]),
    np.array([50., 200., 150., 100.])
]

new_tests = []

for state in small_states:

    bellman_value, bellman_action = bellman_stage2(state)

    full_value, full_actions, count = check_second_stage(state)

    correct = (
        np.isclose(bellman_value, full_value, atol=1e-6)
        and tuple(int(x) for x in bellman_action) in full_actions
    )

    new_tests.append({
        "Состояние": tuple(state),
        "Беллман": bellman_value,
        "Полный перебор": full_value,
        "Управление": tuple(int(x) for x in bellman_action),
        "Проверено управлений": count,
        "Успешно": correct
    })

new_results = pd.DataFrame(new_tests)

display(new_results)

print(
    "Пройдено дополнительных тестов:",
    new_results["Успешно"].sum(),
    "из",
    len(new_results)
)

assert new_results["Успешно"].all()

,Состояние,Беллман,Полный перебор,Управление,Проверено управлений,Успешно
0,"(50.0, 200.0, 150.0, 50.0)",458.4508,458.4508,"(0, 0, 0)",2,True
1,"(50.0, 200.0, 150.0, 100.0)",508.4508,508.4508,"(0, 0, 0)",4,True


Пройдено дополнительных тестов: 2 из 2


### Проверка нулевой доходности и комиссий

In [37]:
import types

# Проверяем случай без изменения стоимости и без комиссий
test_commissions = np.zeros(3)
test_changes = np.ones((3, 3, 3))
test_average = np.ones(3)

# Создаём отдельное окружение для теста
test_globals = globals().copy()

test_globals["commissions"] = test_commissions
test_globals["price_changes"] = test_changes
test_globals["average_last"] = test_average

# Используем Python-версии функций основного алгоритма
function_names = [
    "total_capital",
    "make_trades",
    "trades_allowed",
    "market_change",
    "minimum_packages",
    "expected_last_capital",
    "trade_limits",
    "bellman_stage3",
    "bellman_stage2"
]

for name in function_names:
    original = globals()[name].py_func

    test_globals[name] = types.FunctionType(
        original.__code__,
        test_globals,
        name,
        original.__defaults__,
        original.__closure__
    )

# Небольшое состояние второго этапа
test_state = np.array([50.0, 200.0, 150.0, 50.0])

expected_value = sum(test_state)

# Запускаем исходную логику Беллмана с новыми параметрами
actual_value, actual_action = test_globals["bellman_stage2"](test_state)

correct = np.isclose(actual_value, expected_value)

print("Начальный капитал:", expected_value)
print("Результат Беллмана:", actual_value)
print("Найденное управление:", tuple(int(x) for x in actual_action))
print("Проверка пройдена:", correct)

assert correct

Начальный капитал: 450.0
Результат Беллмана: 450.0
Найденное управление: (0, 0, 0)
Проверка пройдена: True


### Время выполнения и оценка памяти



In [38]:
import time
import os
import psutil

process = psutil.Process(os.getpid())

memory_before = process.memory_info().rss / 1024**2

start = time.perf_counter()

value, action = bellman_stage1(initial_portfolio)

execution_time = time.perf_counter() - start

memory_after = process.memory_info().rss / 1024**2

print(f"Время выполнения: {execution_time:.3f} с")
print(f"Память до выполнения: {memory_before:.2f} МБ")
print(f"Память после выполнения: {memory_after:.2f} МБ")
print(f"Изменение памяти: {memory_after - memory_before:.2f} МБ")

assert np.isclose(value, optimal_capital, atol=1e-6)

Время выполнения: 10.019 с
Память до выполнения: 268.34 МБ
Память после выполнения: 268.34 МБ
Изменение памяти: 0.00 МБ


In [39]:
n_first = len(first_array)

n_second_calls = n_first * 3

actions_memory = first_array.nbytes
values_memory = first_scores.nbytes

total_memory = actions_memory + values_memory

print("Допустимых управлений первого этапа:", n_first)
print("Вызовов второго этапа:", n_second_calls)

print("\nПамять массивов:")
print("Управления:", actions_memory, "байт")
print("Значения капитала:", values_memory, "байт")
print("Всего:", total_memory, "байт")
print("Всего:", round(total_memory / 1024, 2), "КиБ")

assert n_first == 1502
assert n_second_calls == 4506

Допустимых управлений первого этапа: 1502
Вызовов второго этапа: 4506

Память массивов:
Управления: 36048 байт
Значения капитала: 12016 байт
Всего: 48064 байт
Всего: 46.94 КиБ


### Бонусное задание. Дерево исходов


In [40]:
situations = ["Б", "Н", "О"]

nodes = {}
branches = []
final_results = []

start_capital = total_capital(initial_portfolio)


def trade_fee(trades):
    fee = 0
    for i in range(3):
        fee += abs(trades[i]) * package_sizes[i] * commissions[i]
    return fee


nodes["start"] = (initial_portfolio, first_trades)

after_first = make_trades(initial_portfolio, first_trades)

for i in range(3):
    state2 = market_change(after_first, 0, i)
    value2, action2 = bellman_stage2(state2)

    node2 = f"2_{i}"
    nodes[node2] = (state2, action2)
    branches.append(("start", node2, situations[i], probabilities[0, i]))

    after_second = make_trades(state2, action2)

    for j in range(3):
        state3 = market_change(after_second, 1, j)
        value3, action3 = bellman_stage3(state3)

        node3 = f"3_{i}_{j}"
        nodes[node3] = (state3, action3)
        branches.append((node2, node3, situations[j], probabilities[1, j]))

        after_third = make_trades(state3, action3)

        for k in range(3):
            final_state = market_change(after_third, 2, k)

            capital = total_capital(final_state)
            income = capital - start_capital
            return_percent = income / start_capital * 100

            chance = (
                probabilities[0, i]
                * probabilities[1, j]
                * probabilities[2, k]
            )

            fees = (
                trade_fee(first_trades)
                + trade_fee(action2)
                + trade_fee(action3)
            )

            sequence = situations[i] + situations[j] + situations[k]
            node4 = f"4_{i}_{j}_{k}"

            nodes[node4] = (final_state, None)
            branches.append((node3, node4, situations[k], probabilities[2, k]))

            final_results.append({
                "Исход": sequence,
                "Вероятность": chance,
                "Комиссии": fees,
                "Капитал": capital,
                "Доход": income,
                "Доходность, %": return_percent
            })

print("Узлов дерева:", len(nodes))
print("Переходов:", len(branches))
print("Конечных исходов:", len(final_results))

Узлов дерева: 40
Переходов: 39
Конечных исходов: 27


In [41]:
results = pd.DataFrame(final_results)

expected_capital = (
    results["Капитал"] * results["Вероятность"]
).sum()

expected_income = (
    results["Доход"] * results["Вероятность"]
).sum()

expected_return = (
    results["Доходность, %"] * results["Вероятность"]
).sum()

loss_probability = results.loc[
    results["Доход"] < 0, "Вероятность"
].sum()

print("Ожидаемый капитал:", round(expected_capital, 4))
print("Ожидаемый доход:", round(expected_income, 4))
print("Ожидаемая доходность:", round(expected_return, 2), "%")
print("Вероятность убытка:", round(loss_probability * 100, 2), "%")
print("Минимальный доход:", round(results["Доход"].min(), 3))
print("Максимальный доход:", round(results["Доход"].max(), 3))

assert len(results) == 27
assert np.isclose(results["Вероятность"].sum(), 1)
assert np.isclose(expected_capital, optimal_capital)

display(results.round(3))

Ожидаемый капитал: 2033.9889
Ожидаемый доход: 133.9889
Ожидаемая доходность: 7.05 %
Вероятность убытка: 19.0 %
Минимальный доход: -366.74
Максимальный доход: 489.684


,Исход,Вероятность,Комиссии,Капитал,Доход,"Доходность, %"
0,БББ,0.072,54.0,2389.684,489.684,25.773
1,ББН,0.072,54.0,2256.173,356.173,18.746
2,ББО,0.036,54.0,2163.860,263.860,13.887
3,БНБ,0.048,54.0,2223.263,323.263,17.014
4,БНН,0.048,54.0,2105.668,205.668,10.825
5,БНО,0.024,54.0,2026.925,126.925,6.680
6,БОБ,0.120,55.0,2109.340,209.340,11.018
7,БОН,0.120,55.0,2000.500,100.500,5.289
8,БОО,0.060,55.0,1924.680,24.680,1.299
9,НББ,0.036,50.0,2192.272,292.272,15.383


In [42]:
import sys
print(sys.executable)

/usr/local/bin/python3


In [43]:
%pip install graphviz


[notice] A new release of pip is available: 26.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [44]:
from graphviz import Digraph

graph = Digraph("investment", format="pdf")

graph.attr(rankdir="LR")
graph.attr("node", shape="box", fontsize="9")
graph.attr("edge", fontsize="9")


def portfolio_text(state):
    return (
        f"ЦБ1: {state[0]:.2f}\n"
        f"ЦБ2: {state[1]:.2f}\n"
        f"Депозит: {state[2]:.2f}\n"
        f"Деньги: {state[3]:.2f}"
    )


# Добавляем состояния
for name, (state, action) in nodes.items():

    label = portfolio_text(state)

    if action is not None:
        action_text = tuple(int(x) for x in action)
        label += f"\nДействие: {action_text}"

    if name.startswith("4_"):
        label += f"\nКапитал: {total_capital(state):.2f}"

    graph.node(name, label)


# Соединяем состояния переходами
for start, end, situation, probability in branches:
    graph.edge(
        start,
        end,
        label=f"{situation}: {probability:.2f}"
    )


graph.render("bonus_tree", cleanup=True)

print("Дерево сохранено в bonus_tree.pdf")

Дерево сохранено в bonus_tree.pdf
